In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
# Load the dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [3]:
# Normalize the pixel values
x_train, x_test = x_train / 255.0, x_test / 255.0

In [4]:
# Reshape the dataset
# (# of images, height, width, # of colour channels)
# NOTE: -1 tells Python to figure out the number of images automatically
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

In [5]:
# Create a model

model = tf.keras.Sequential([

    # First layer
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),

    # Second layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    # Dense layer for classification (10 outputs in total)
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

2026-05-16 11:29:08.637049: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-05-16 11:29:08.637082: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-05-16 11:29:08.637089: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2026-05-16 11:29:08.637140: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-16 11:29:08.637161: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [8]:
# Summary of the model
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 64)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 1600)              0         
                                                                 
 dense (Dense)               (None, 64)                1

In [11]:
# Train the model
history = model.fit(x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[callback])

Epoch 1/10
844/844 [==============================] - 4s 5ms/step - loss: 0.1305 - accuracy: 0.9707 - val_loss: 0.1404 - val_accuracy: 0.9802
Epoch 2/10
844/844 [==============================] - 4s 5ms/step - loss: 0.1866 - accuracy: 0.9700 - val_loss: 0.1078 - val_accuracy: 0.9870
Epoch 3/10
844/844 [==============================] - 4s 5ms/step - loss: 0.2494 - accuracy: 0.9715 - val_loss: 0.1713 - val_accuracy: 0.9863
Epoch 4/10
844/844 [==============================] - 4s 5ms/step - loss: 0.3941 - accuracy: 0.9707 - val_loss: 0.3147 - val_accuracy: 0.9847
Epoch 5/10
844/844 [==============================] - 4s 5ms/step - loss: 0.6084 - accuracy: 0.9696 - val_loss: 0.4228 - val_accuracy: 0.9855


In [12]:
print(f"Number of epoch are run: {len(history.history['loss'])}")

Number of epoch are run: 5


In [13]:
# Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

313/313 - 1s - loss: 0.1033 - accuracy: 0.9848 - 751ms/epoch - 2ms/step

Test Accuracy: 98.48%


In [15]:
print("Max value in x_train:", x_train.max())
print("Min value in x_train:", x_train.min())

Max value in x_train: 1.0
Min value in x_train: 0.0


In [16]:
# Save the model and its parameters
model.save('digit_recognition_model.h5')

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
